# API quickstart — Multicellular Transformer

Scanpy-style entry points: one `cci.load(...)`, then `cci.tl.*` / `cci.pl.*` / `cci.pp.*`.

**Cells are illustrative (unexecuted).** Run against a local model directory.

## 1. Load a model directory

In [ ]:
import cell_cell_interactions as cci

# Directory layout expected:
#   /models/mct_v1/
#     checkpoints/model.ckpt   (or best.ckpt / last.ckpt / any *.ckpt)
#     gene2idx.json            (or gene_symbol2idx.json)
#     cell_type_input_order.json
#     tissue2idx.json          (optional)
#     cell_type_map.json       (optional)

mct = cci.load("/braid/andera29/models/mct/registry/mct_v1", device="cuda")
print(mct.config)
print("n vocab genes:", len(mct.gene2idx))

## 2. Zero-shot gene embeddings — DataFrame in, DataFrame out

In [ ]:
emb = mct.gene_embeddings()            # (n_genes x gene_emb_dim)
print(emb.shape)
emb.loc[["EGFR", "TP53", "MYC"]].head()

## 3. Analysis with `cci.tl`

In [ ]:
# Rank neighbors of a query gene
ranking = cci.tl.rank_similar_genes(emb, "EGFR", top_n=20)
print(ranking.head(10))

# Score named pairs (e.g. receptor-ligand)
pairs = [("EGFR", "EGF"), ("KIT", "KITLG"), ("PDGFRA", "PDGFA")]
scores = cci.tl.pairwise_cosine_similarity(emb, pairs=pairs)
print(scores)

# Global k-NN gene graph
nbrs = cci.tl.gene_neighbors(emb, n_neighbors=15)
print(nbrs[nbrs["gene"] == "EGFR"].head())

## 4. Plotting with `cci.pl` — returns Axes, no `plt.show()`

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
cci.pl.similarity_ranking(ranking, top_n=15, ax=axes[0], title="EGFR neighbors")
cci.pl.similarity_heatmap(emb, genes=["EGFR", "ERBB2", "ERBB3", "ERBB4"], ax=axes[1])
fig.tight_layout()

## 5. Context-aware embeddings from your AnnData

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("/path/to/your.h5ad")

# Preprocessing helpers (optional but recommended)
adata = cci.pp.filter_to_model_vocabulary(adata, mct.gene2idx)
adata = cci.pp.harmonize_cell_types(
    adata,
    cell_type_key="cell_type",
    mapping=mct.index.cell_type_map,     # None if unavailable
)

# One call: pseudobulk -> normalize -> forward -> per-sample DataFrames
sample_embs = mct.embed(
    adata,
    cell_type_key="cell_type",
    sample_key="donor_id",
    tissue_key="tissue",                  # optional; used if model has a tissue vocab
)

print("samples:", list(sample_embs)[:3])
example_donor = next(iter(sample_embs))
print(sample_embs[example_donor].shape)
sample_embs[example_donor].loc[["EGFR", "TP53"]].head()

## 6. Batch effects

Fisher-transform pair scores and orthogonalize against `study` while keeping `tissue`.

In [ ]:
# scores must carry study + tissue columns per row
adjusted = cci.tl.regress_out(scores_with_meta, regress_out="study", keep=["tissue"])
adjusted[["pair", "score", "score_adj"]].head()

## 7. Sample-level and DepMap KO workflows (utils)

In [ ]:
from cell_cell_interactions.utils.sample_embeddings.api import (
    collect_sample_embeddings, collect_sample_neighbors,
)
from cell_cell_interactions.utils.depmap.api import run_gene_ko_effect, DEPMAP_DISEASE_MAP

# Example wiring (fill in with your datamodule / dataloader):
# stacked, meta = collect_sample_embeddings(mct, my_dataloader, output_dir="outputs/")
# expr, rel_expr = run_gene_ko_effect(
#     model=mct,
#     dataloader=my_dataloader,
#     tumor_cell_type="Malignant cells",
#     genes_to_knock_out=["EGFR", "KRAS"],
#     name2idx={"EGFR": mct.gene2idx["EGFR"], "KRAS": mct.gene2idx["KRAS"]},
#     output_dir="outputs/ko",
# )